<a href="https://colab.research.google.com/github/k242565-art/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k242565-art/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/k242565-art/flyrank-ml-internship.git
%cd flyrank-ml-internship
!find . -maxdepth 3 -type f | head -100

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 126 (delta 37), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 1.85 MiB | 12.47 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
./Copy_of_01_first_look_and_discovery.ipynb
./scripts/02_baseline_score.py
./scripts/run_all.py
./scripts/05_build_pdf_report.py
./scripts/03_train_model.py
./scripts/ml_utils.py
./scripts/04_evaluate_and_export.py
./scripts/01_prepare_features.py
./.github/workflows/data-path-smoke.yml
./.github/workflows/smoke-test.yml
./.github/workflows/personalize.yml
./AGENTS.md
./submission/paper_url.txt
./submission/README.md
./outputs/charts/top_reason_codes.svg
./outputs/charts/top_feature_importance.svg
./outputs/charts/trend_distribution.svg
./outputs/charts/action_mix

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Signal: days_since_last_update

Verdict: MIXED

The relationship between content staleness and performance is not consistently monotonic. Content updated within 91–180 days has the highest average sessions, while the oldest content (181+ days) performs substantially worse. However, the oldest buckets contain very few observations (n=169 and n=5), making those estimates less reliable. The signal may still be useful for refresh prioritization, but by itself it does not strongly confirm that older content always performs worse.
Signal: trend_pct

Verdict: MIXED

Traffic trend does not show a simple relationship with average sessions. Pages in the Decline bucket have the highest average sessions, followed by Growth. Both Strong Decline and Strong Growth have lower average sessions. This suggests trend alone is not sufficient for refresh prioritization and should be combined with other signals such as staleness or search opportunity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os
print(os.getcwd())
!ls

for root, dirs, files in os.walk("."):
    for f in files:
        if f.endswith(".parquet") or f.endswith(".csv") or f.endswith(".json"):
            print(os.path.join(root, f))

for root, dirs, files in os.walk("."):
    for f in files:
        if "w04_baseline_score.ipynb" in f:
            print(os.path.join(root, f))
import os
print(os.getcwd())
!find . -type f | head -100

import pandas as pd

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()
df.columns.tolist()

import pandas as pd

df["update_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0,30,90,180,365,10000],
    labels=[
        "0-30",
        "31-90",
        "91-180",
        "181-365",
        "365+"
    ]
)

signal1 = (
    df.groupby("update_bucket", observed=False)
      .agg(
          avg_sessions=("sessions_90d","mean"),
          n=("sessions_90d","count")
      )
)

signal1

df["trend_bucket"] = pd.cut(
    df["trend_pct"],
    bins=[-100,-25,0,25,1000],
    labels=[
        "Strong Decline",
        "Decline",
        "Growth",
        "Strong Growth"
    ]
)

signal2 = (
    df.groupby("trend_bucket", observed=False)
      .agg(
          avg_sessions=("sessions_90d","mean"),
          n=("sessions_90d","count")
      )
)

signal2

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
02_your_first_readable_model.ipynb	   README.md
AGENTS.md				   requirements.txt
CLAUDE.md				   scripts
Copy_of_01_first_look_and_discovery.ipynb  SETUP.md
data					   skills
DATA_USE.md				   submission
docs					   w01_research_question.ipynb
GUIDE.md				   w02_ml_task_framing.ipynb
LICENSE					   w03_data_contract.ipynb
notebooks				   work
outputs
./outputs/refresh_queue_sample.csv
./data/raw/content_refresh_anonymized.csv
./work/notebooks/w04_baseline_score.ipynb
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
./Copy_of_01_first_look_and_discovery.ipynb
./scripts/02_baseline_score.py
./scripts/run_all.py
./scripts/05_build_pdf_report.py
./scripts/03_train_model.py
./scripts/ml_utils.py
./scripts/04_evaluate_and_export.py
./scripts/01_prepare_features.py
./.github/workflows/data-path-smoke.yml
./.github/workflows/smoke-test.yml
./.github/workflows/personalize.yml
./AGENTS.md
./subm

,avg_sessions,n
trend_bucket,,
Strong Decline,35.238679,13868
Decline,68.393871,4895
Growth,62.192355,2433
Strong Growth,28.027679,3938


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["refresh_score"] = (
    df["days_since_last_update"] * 0.7
    +
    (-df["trend_pct"].clip(upper=0)) * 0.3
)
df["reason_code"] = "STALE_DECLINING_CONTENT"

queue = (
    df.sort_values(
        "refresh_score",
        ascending=False
    )
    [
        [
            "content_id",
            "refresh_score",
            "reason_code",
            "days_since_last_update",
            "trend_pct",
            "sessions_90d"
        ]
    ]
)

queue.head()
import os
print(os.getcwd())
!ls
!find . -type d | grep outputs
import os

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
02_your_first_readable_model.ipynb	   README.md
AGENTS.md				   requirements.txt
CLAUDE.md				   scripts
Copy_of_01_first_look_and_discovery.ipynb  SETUP.md
data					   skills
DATA_USE.md				   submission
docs					   w01_research_question.ipynb
GUIDE.md				   w02_ml_task_framing.ipynb
LICENSE					   w03_data_contract.ipynb
notebooks				   work
outputs
./outputs
./outputs/charts


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
Action:
REFRESH_CONTENT

Why:
The page has not been updated for a long period and is showing a negative trend.

What would make it wrong:
The topic may be evergreen or seasonal, meaning refresh is unnecessary.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.head(20)[[
    "content_id",
    "days_since_last_update",
    "trend_pct",
    "refresh_score"
]]

,content_id,days_since_last_update,trend_pct,refresh_score
29384,content_f6fdf87348f6,373,-100.0,291.10
24216,content_1b4ec72dafd4,372,-100.0,290.40
26242,content_55a5b1c46474,373,-88.5,287.65
6962,content_f01216059a6a,335,-71.0,255.80
8631,content_e2b702f4f92b,334,-72.7,255.61
7509,content_7a888d3d99c8,313,-100.0,249.10
18841,content_94991fe6268c,313,-100.0,249.10
21984,content_02b0d6e30129,313,-95.6,247.78
22860,content_ab18b5811c02,305,-100.0,243.50
24557,content_84d12054c0c0,304,-100.0,242.80


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
These rows scored low because they are either recently updated, showing positive traffic trends, or both. The rule therefore assigns lower refresh priority.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.tail(10)

,content_id,refresh_score,reason_code,days_since_last_update,trend_pct,sessions_90d
29919,content_0934cd438dd6,NaN,STALE_DECLINING_CONTENT,8,NaN,22
29920,content_f452475bff46,NaN,STALE_DECLINING_CONTENT,20,NaN,1
29934,content_cd850fb019b1,NaN,STALE_DECLINING_CONTENT,20,NaN,1
29960,content_b1d45033b059,NaN,STALE_DECLINING_CONTENT,20,NaN,2
29965,content_a3af3b8346d8,NaN,STALE_DECLINING_CONTENT,6,NaN,22
29968,content_179533212cd0,NaN,STALE_DECLINING_CONTENT,20,NaN,17
29971,content_92a5d2709aa9,NaN,STALE_DECLINING_CONTENT,8,NaN,24
29981,content_2dfd17269502,NaN,STALE_DECLINING_CONTENT,20,NaN,1
29992,content_23dce6a656e6,NaN,STALE_DECLINING_CONTENT,20,NaN,1
29995,content_c322796023c8,NaN,STALE_DECLINING_CONTENT,20,NaN,3


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.